In [52]:
import torch
print(torch.__version__)

2.8.0+cu128


In [53]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import pickle
import numpy as np
from tqdm import tqdm
from torch_geometric.data import HeteroData
from torch_geometric.transforms import ToUndirected, AddSelfLoops
from torch_geometric.nn import HeteroConv, GATConv, Linear, SAGEConv
from torch_geometric.loader import LinkNeighborLoader
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
import random

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)


In [54]:
"""
Drug Repositioning Pipeline with GNN - MINI-BATCH VERSION
==========================================================
Memory-efficient pipeline using mini-batch training with neighbor sampling
"""


# ============================================================================

# 1. DATA LOADING
# ============================================================================

print("="*80)
print("STEP 1: Loading Data")
print("="*80)

with open('bio_graph_raw.pkl', 'rb') as f:
    G = pickle.load(f)

# Separate node types
chemicals = sorted([n for n, d in G.nodes(data=True) if d['node_type'] == 'chemical'])
diseases  = sorted([n for n, d in G.nodes(data=True) if d['node_type'] == 'disease'])
genes     = sorted([n for n, d in G.nodes(data=True) if d['node_type'] == 'gene'])

chem_idx = {c: i for i, c in enumerate(chemicals)}
disease_idx = {d: i for i, d in enumerate(diseases)}
gene_idx = {g: i for i, g in enumerate(genes)}

print(f"Number of chemicals: {len(chemicals)}")
print(f"Number of diseases: {len(diseases)}")
print(f"Number of genes: {len(genes)}")

# Extract features
chem_features = torch.tensor(
    np.array([G.nodes[c]['x'] for c in chemicals], dtype=np.float32)
)
disease_features = torch.tensor(
    np.array([G.nodes[d]['x'] for d in diseases], dtype=np.float32)
)
gene_features = torch.tensor(
    np.array([G.nodes[g]['x'] for g in genes], dtype=np.float32)
)

print(f"Chemical feature dim: {chem_features.shape[1]}")
print(f"Disease feature dim: {disease_features.shape[1]}")
print(f"Gene feature dim: {gene_features.shape[1]}")


# ============================================================================
# 2. EDGE EXTRACTION
# ============================================================================

print("\n" + "="*80)
print("STEP 2: Extracting Edges")
print("="*80)

chem_gene_edges = []
chem_disease_edges = []
gene_disease_edges = []

for u, v, data in G.edges(data=True):
    if data['edge_type'] == 'chem_gene':
        chem_gene_edges.append([chem_idx[u], gene_idx[v]])
    elif data['edge_type'] == 'chem_disease':
        chem_disease_edges.append([chem_idx[u], disease_idx[v]])
    elif data['edge_type'] == 'gene_disease':
        gene_disease_edges.append([gene_idx[u], disease_idx[v]])

chem_gene_edges = torch.tensor(chem_gene_edges).t().contiguous()
chem_disease_edges = torch.tensor(chem_disease_edges).t().contiguous()
gene_disease_edges = torch.tensor(gene_disease_edges).t().contiguous()


print(f"Chemical-Gene edges: {chem_gene_edges.shape[1]}")
print(f"Chemical-Disease edges: {chem_disease_edges.shape[1]}")
print(f"Gene-Disease edges: {gene_disease_edges.shape[1]}")

# Compute node degrees for degree-preserving negative sampling
print("\nComputing node degrees for hard negative sampling...")
chem_degrees = torch.zeros(len(chemicals), dtype=torch.long)
disease_degrees = torch.zeros(len(diseases), dtype=torch.long)

for i in range(chem_disease_edges.shape[1]):
    chem_degrees[chem_disease_edges[0, i]] += 1
    disease_degrees[chem_disease_edges[1, i]] += 1

print(f"Chemical degree stats - Min: {chem_degrees.min()}, Max: {chem_degrees.max()}, Mean: {chem_degrees.float().mean():.2f}, Median: {chem_degrees.float().median():.2f}")
print(f"Disease degree stats - Min: {disease_degrees.min()}, Max: {disease_degrees.max()}, Mean: {disease_degrees.float().mean():.2f}, Median: {disease_degrees.float().median():.2f}")



STEP 1: Loading Data


Number of chemicals: 13943
Number of diseases: 7239
Number of genes: 28151
Chemical feature dim: 3591
Disease feature dim: 2605
Gene feature dim: 2635

STEP 2: Extracting Edges
Chemical-Gene edges: 703832
Chemical-Disease edges: 2911974
Gene-Disease edges: 34221

Computing node degrees for hard negative sampling...
Chemical degree stats - Min: 0, Max: 5832, Mean: 208.85, Median: 66.00
Disease degree stats - Min: 0, Max: 8370, Mean: 402.26, Median: 90.00


In [55]:

# ============================================================================
# 3. TRAIN/VAL/TEST SPLIT
# ============================================================================

print("\n" + "="*80)
print("STEP 3: Splitting Data (Train/Val/Test)")
print("="*80)

# Train/Val/Test split for chemical-disease edges (our prediction target)
num_edges = chem_disease_edges.shape[1]
perm = torch.randperm(num_edges)

# 70% train, 15% val, 15% test
train_size = int(0.7 * num_edges)
val_size = int(0.15 * num_edges)

train_idx = perm[:train_size]
val_idx = perm[train_size:train_size + val_size]
test_idx = perm[train_size + val_size:]

train_edges = chem_disease_edges[:, train_idx]
val_edges = chem_disease_edges[:, val_idx]
test_edges = chem_disease_edges[:, test_idx]

print(f"Train edges: {train_edges.shape[1]}")
print(f"Val edges: {val_edges.shape[1]}")
print(f"Test edges: {test_edges.shape[1]}")


# ============================================================================
# 4. NEGATIVE SAMPLING
# ============================================================================

print("\n" + "="*80)
print("STEP 4: Generating Negative Samples")
print("="*80)



STEP 3: Splitting Data (Train/Val/Test)
Train edges: 2038381
Val edges: 436796
Test edges: 436797

STEP 4: Generating Negative Samples


In [ ]:

def create_degree_bins(degrees, bin_edges):
    """
    Assign nodes to degree bins based on their degrees.
    Returns a dictionary: bin_id -> list of node indices
    """
    bins = {i: [] for i in range(len(bin_edges) - 1)}
    
    for node_id, deg in enumerate(degrees.tolist()):
        for bin_id in range(len(bin_edges) - 1):
            if bin_edges[bin_id] <= deg < bin_edges[bin_id + 1]:
                bins[bin_id].append(node_id)
                break
        else:  # Handle nodes with degree >= max bin edge
            bins[len(bin_edges) - 2].append(node_id)
    
    return bins


def degree_preserving_negative_sampling(edge_index, chem_degrees, disease_degrees, 
                                       num_neg_samples, bin_edges_chem, bin_edges_disease):
    """
    Generate hard negative samples that match the degree distribution of positive edges.
    
    For each positive edge (chem, disease):
    1. Find degree bins of the chemical and disease
    2. Sample a chemical from the same/similar degree bin
    3. Sample a disease from the same/similar degree bin
    4. Ensure the sampled pair is not in the positive edge set
    """
    # Create set of positive edges for fast lookup
    pos_edges_set = set(map(tuple, edge_index.t().tolist()))
    
    # Create degree bins
    chem_bins = create_degree_bins(chem_degrees, bin_edges_chem)
    disease_bins = create_degree_bins(disease_degrees, bin_edges_disease)
    
    # Get statistics of positive edges
    pos_chem_degrees = chem_degrees[edge_index[0]].numpy()
    pos_disease_degrees = disease_degrees[edge_index[1]].numpy()
    
    neg_edges = []
    neg_chem_degrees = []
    neg_disease_degrees = []
    
    attempts = 0
    max_attempts = num_neg_samples * 20  # More attempts for degree matching
    
    # Sample negatives to match positive edge degree distribution
    num_pos_edges = edge_index.shape[1]
    
    while len(neg_edges) < num_neg_samples and attempts < max_attempts:
        # Randomly select a positive edge to mimic its degree pattern
        pos_idx = random.randint(0, num_pos_edges - 1)
        target_chem_deg = chem_degrees[edge_index[0, pos_idx]].item()
        target_disease_deg = disease_degrees[edge_index[1, pos_idx]].item()
        
        # Find which bins these degrees belong to
        chem_bin_id = None
        for bin_id in range(len(bin_edges_chem) - 1):
            if bin_edges_chem[bin_id] <= target_chem_deg < bin_edges_chem[bin_id + 1]:
                chem_bin_id = bin_id
                break
        if chem_bin_id is None:
            chem_bin_id = len(bin_edges_chem) - 2
        
        disease_bin_id = None
        for bin_id in range(len(bin_edges_disease) - 1):
            if bin_edges_disease[bin_id] <= target_disease_deg < bin_edges_disease[bin_id + 1]:
                disease_bin_id = bin_id
                break
        if disease_bin_id is None:
            disease_bin_id = len(bin_edges_disease) - 2
        
        # Sample from the same bins (with fallback to adjacent bins)
        chem_candidates = chem_bins[chem_bin_id].copy()
        disease_candidates = disease_bins[disease_bin_id].copy()
        
        # Fallback: if bin is empty, use adjacent bins
        if not chem_candidates:
            for offset in [1, -1, 2, -2]:
                neighbor_bin = chem_bin_id + offset
                if 0 <= neighbor_bin < len(bin_edges_chem) - 1 and chem_bins[neighbor_bin]:
                    chem_candidates = chem_bins[neighbor_bin].copy()
                    break
        
        if not disease_candidates:
            for offset in [1, -1, 2, -2]:
                neighbor_bin = disease_bin_id + offset
                if 0 <= neighbor_bin < len(bin_edges_disease) - 1 and disease_bins[neighbor_bin]:
                    disease_candidates = disease_bins[neighbor_bin].copy()
                    break
        
        if chem_candidates and disease_candidates:
            src = random.choice(chem_candidates)
            dst = random.choice(disease_candidates)
            
            if (src, dst) not in pos_edges_set:
                neg_edges.append([src, dst])
                neg_chem_degrees.append(chem_degrees[src].item())
                neg_disease_degrees.append(disease_degrees[dst].item())
        
        attempts += 1
    
    if len(neg_edges) < num_neg_samples:
        print(f"  Warning: Could only generate {len(neg_edges)} negative samples (requested {num_neg_samples})")
    
    # Print degree statistics comparison
    print(f"  Positive edges - Chem degree mean: {pos_chem_degrees.mean():.2f}, Disease degree mean: {pos_disease_degrees.mean():.2f}")
    if neg_chem_degrees:
        print(f"  Negative edges - Chem degree mean: {np.mean(neg_chem_degrees):.2f}, Disease degree mean: {np.mean(neg_disease_degrees):.2f}")
    
    return torch.tensor(neg_edges).t().contiguous() if neg_edges else torch.empty((2, 0), dtype=torch.long)

# Generate negative samples
num_chemicals = len(chemicals)
num_diseases = len(diseases)

# Define degree bins based on the power-law distribution
# Bins: [0-10], [10-50], [50-100], [100-500], [500-2000], [2000+]
bin_edges_chem = [0, 10, 50, 100, 500, 2000, float('inf')]
bin_edges_disease = [0, 10, 50, 100, 500, 2000, float('inf')]


# CRITICAL: Negative sampling must exclude different edges for each split to prevent data leakage

train_neg_edges = degree_preserving_negative_sampling(
    train_edges, chem_degrees, disease_degrees, 
    train_edges.shape[1], bin_edges_chem, bin_edges_disease
)


train_val_edges = torch.cat([train_edges, val_edges], dim=1)
val_neg_edges = degree_preserving_negative_sampling(
    train_val_edges, chem_degrees, disease_degrees, 
    val_edges.shape[1], bin_edges_chem, bin_edges_disease
)

all_edges = torch.cat([train_edges, val_edges, test_edges], dim=1)
test_neg_edges = degree_preserving_negative_sampling(
    all_edges, chem_degrees, disease_degrees, 
    10*test_edges.shape[1], bin_edges_chem, bin_edges_disease
)




In [57]:


# ============================================================================
# 5. BUILD HETEROGENEOUS GRAPH
# ============================================================================

print("\n" + "="*80)
print("STEP 5: Building Heterogeneous Graph")
print("="*80)

# Build HeteroData
data = HeteroData()

data['chemical'].x = chem_features
data['disease'].x = disease_features
data['gene'].x = gene_features

# Use all auxiliary edges
data['chemical', 'chem_gene', 'gene'].edge_index = chem_gene_edges
data['gene', 'gene_disease', 'disease'].edge_index = gene_disease_edges

# Use only TRAINING chem-disease edges for message passing
data['chemical', 'chem_disease', 'disease'].edge_index = train_edges

# Store edge labels for link prediction
data['chemical', 'chem_disease', 'disease'].edge_label_index = torch.cat([train_edges, train_neg_edges], dim=1)
data['chemical', 'chem_disease', 'disease'].edge_label = torch.cat([
    torch.ones(train_edges.shape[1]),
    torch.zeros(train_neg_edges.shape[1])
])

# Make the graph undirected
data = ToUndirected()(data)
data = AddSelfLoops()(data)

print(data)




STEP 5: Building Heterogeneous Graph
HeteroData(
  chemical={ x=[13943, 3591] },
  disease={ x=[7239, 2605] },
  gene={ x=[28151, 2635] },
  (chemical, chem_gene, gene)={ edge_index=[2, 703832] },
  (gene, gene_disease, disease)={ edge_index=[2, 34221] },
  (chemical, chem_disease, disease)={
    edge_index=[2, 2038381],
    edge_label_index=[2, 18345429],
    edge_label=[18345429],
  },
  (gene, rev_chem_gene, chemical)={ edge_index=[2, 703832] },
  (disease, rev_gene_disease, gene)={ edge_index=[2, 34221] },
  (disease, rev_chem_disease, chemical)={ edge_index=[2, 2038381] }
)


In [58]:
import pickle
with open('findata.pkl', 'wb') as file:
    pickle.dump(data, file)

In [59]:
import pickle
with open('findata.pkl', 'rb') as file:
    data = pickle.load(file)

In [60]:

# ============================================================================
# 6. MINI-BATCH DATA LOADERS
# ============================================================================

print("\n" + "="*80)
print("STEP 6: Creating Mini-Batch Data Loaders")
print("="*80)

# Create mini-batch loaders using LinkNeighborLoader
# This samples K-hop neighbors for each edge in the batch

train_loader = LinkNeighborLoader(
    data,
    num_neighbors=[50,25,10],  # Sample 10 neighbors in 1st hop, 5 in 2nd hop
    edge_label_index=(('chemical', 'chem_disease', 'disease'), 
                      torch.cat([train_edges, train_neg_edges], dim=1)),
    edge_label=torch.cat([
        torch.ones(train_edges.shape[1]),
        torch.zeros(train_neg_edges.shape[1])
    ]),
    batch_size=2097152,  # Process 512 edges per batch
    shuffle=True,
    num_workers=0,
)

# Validation loader
val_edge_label_index = torch.cat([val_edges, val_neg_edges], dim=1)
val_edge_label = torch.cat([
    torch.ones(val_edges.shape[1]),
    torch.zeros(val_neg_edges.shape[1])
])

val_loader = LinkNeighborLoader(
    data,
    num_neighbors=[50,25,10],
    edge_label_index=(('chemical', 'chem_disease', 'disease'), val_edge_label_index),
    edge_label=val_edge_label,
    batch_size=2097152,
    shuffle=False,
    num_workers=0,
)

# Test loader
test_edge_label_index = torch.cat([test_edges, test_neg_edges], dim=1)
test_edge_label = torch.cat([
    torch.ones(test_edges.shape[1]),
    torch.zeros(test_neg_edges.shape[1])
])

test_loader = LinkNeighborLoader(
    data,
    num_neighbors=[50,25,10],
    edge_label_index=(('chemical', 'chem_disease', 'disease'), test_edge_label_index),
    edge_label=test_edge_label,
    batch_size=2097152,
    shuffle=False,
    num_workers=0,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")



STEP 6: Creating Mini-Batch Data Loaders
Train batches: 9
Val batches: 2
Test batches: 3


In [ ]:


# ============================================================================
# 7. IMPROVED MODEL DEFINITION
# ============================================================================

print("\n" + "="*80)
print("STEP 7: Defining IMPROVED Heterogeneous GNN Model")
print("="*80)

class HeteroGNN(nn.Module):    
    def __init__(self, hidden_channels=128, out_channels=64, num_layers=3, dropout=0.2):
        super().__init__()
        
        self.dropout = dropout
        self.convs = nn.ModuleList()

        for _ in range(num_layers):
            self.convs.append(HeteroConv({
                ('chemical', 'chem_gene', 'gene'): SAGEConv((-1, -1), hidden_channels),
                ('gene', 'rev_chem_gene', 'chemical'): SAGEConv((-1, -1), hidden_channels),
                ('chemical', 'chem_disease', 'disease'): SAGEConv((-1, -1), hidden_channels),
                ('disease', 'rev_chem_disease', 'chemical'): SAGEConv((-1, -1), hidden_channels),
                ('gene', 'gene_disease', 'disease'): SAGEConv((-1, -1), hidden_channels),
                ('disease', 'rev_gene_disease', 'gene'): SAGEConv((-1, -1), hidden_channels),
            }, aggr='mean'))

        # Final projection layers
        self.lin_chemical = Linear(hidden_channels, out_channels)
        self.lin_disease = Linear(hidden_channels, out_channels)

        # 🔥 MLP Decoder (major improvement over dot product)
        self.decoder = nn.Sequential(
            nn.Linear(out_channels * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x_dict, edge_index_dict):
        for conv in self.convs:
            x_dict = conv(x_dict, edge_index_dict)
            x_dict = {k: F.relu(x) for k, x in x_dict.items()}
            x_dict = {k: F.dropout(x, p=self.dropout, training=self.training)
                      for k, x in x_dict.items()}

        z_chemical = self.lin_chemical(x_dict['chemical'])
        z_disease = self.lin_disease(x_dict['disease'])

        return z_chemical, z_disease

    def decode(self, z_chemical, z_disease, edge_label_index):
        src = z_chemical[edge_label_index[0]]
        dst = z_disease[edge_label_index[1]]
        edge_feat = torch.cat([src, dst], dim=-1)
        return self.decoder(edge_feat).squeeze()


# Initialize improved model
device = torch.device('cuda' if torch.cuda.is_available() else '"cpu')
model = HeteroGNN().to(device)

# print(f"Model parameters: {sum(p.numel() for p in model.parameters())}")
print(f"Device: {device}")



STEP 7: Defining IMPROVED Heterogeneous GNN Model
Device: cuda


In [83]:

# ============================================================================
# 8. TRAINING SETUP
# ============================================================================

# ============================================================================
# 8. TRAINING SETUP (Improved)
# ============================================================================

print("\n" + "="*80)
print("STEP 8: Setting up Training (Improved)")
print("="*80)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.00005,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=5,
    # verbose=True
)

criterion = nn.BCEWithLogitsLoss()


# ============================================================================
# 9. TRAINING AND EVALUATION FUNCTIONS
# ============================================================================

def train_epoch(model, loader, optimizer, criterion, device):
    """Train the model for one epoch using mini-batches"""
    model.train()
    total_loss = 0
    total_examples = 0
    
    for batch in tqdm(loader, desc="Training"):
        batch = batch.to(device)
        optimizer.zero_grad()
        
        # Forward pass
        z_chemical, z_disease = model(batch.x_dict, batch.edge_index_dict)
        
        # Get edge predictions for this batch
        edge_label_index = batch['chemical', 'chem_disease', 'disease'].edge_label_index
        edge_label = batch['chemical', 'chem_disease', 'disease'].edge_label
        
        # Compute scores
        pred = model.decode(z_chemical, z_disease, edge_label_index)
        
        # Compute loss
        loss = criterion(pred, edge_label)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * edge_label.size(0)
        total_examples += edge_label.size(0)
    
    return total_loss / total_examples


@torch.no_grad()
def evaluate(model, loader, device):
    """Evaluate the model using mini-batches"""
    model.eval()
    
    all_preds = []
    all_labels = []
    
    for batch in tqdm(loader, desc="Evaluating"):
        batch = batch.to(device)
        
        # Forward pass
        z_chemical, z_disease = model(batch.x_dict, batch.edge_index_dict)
        
        # Get edge predictions for this batch
        edge_label_index = batch['chemical', 'chem_disease', 'disease'].edge_label_index
        edge_label = batch['chemical', 'chem_disease', 'disease'].edge_label
        
        # Compute scores
        pred = model.decode(z_chemical, z_disease, edge_label_index)
        pred = torch.sigmoid(pred)
        
        all_preds.append(pred.cpu())
        all_labels.append(edge_label.cpu())
    
    # Concatenate all predictions and labels
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    
    # Calculate metrics
    auc = roc_auc_score(all_labels, all_preds)
    ap = average_precision_score(all_labels, all_preds)
    
    return auc, ap



STEP 8: Setting up Training (Improved)


In [84]:


# ============================================================================
# 10. TRAINING LOOP
# ============================================================================
# ====================
# ========================================================
# 10. TRAINING LOOP (Improved Early Stopping)
# ============================================================================

print("\n" + "="*80)
print("STEP 9: Training the IMPROVED Model")
print("="*80)
model.load_state_dict(torch.load('best_model_minibatch (6).pt'))

num_epochs = 5
best_val_auc = 0
patience = 100
patience_counter = 0
losses = []
rocs = []
for epoch in range(1, num_epochs + 1):

    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_auc, val_ap = evaluate(model, val_loader, device)

    scheduler.step(val_auc)

    print(f'Epoch {epoch:03d} | '
          f'Loss: {train_loss:.4f} | '
          f'Val AUC: {val_auc:.4f} | '
          f'Val AP: {val_ap:.4f}')
    losses.append(train_loss)
    rocs.append(val_auc)
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model_minibatch.pt')
        print("  → New best model saved!")
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f'\nEarly stopping at epoch {epoch}')
        break

print(f'\nBest validation AUC: {best_val_auc:.4f}')



STEP 9: Training the IMPROVED Model


Training:   0%|          | 0/9 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 2/2 [00:00<00:00,  3.26it/s]


Epoch 001 | Loss: 0.5010 | Val AUC: 0.8691 | Val AP: 0.4485
  → New best model saved!


Evaluating: 100%|██████████| 2/2 [00:00<00:00,  3.35it/s]


...

Epoch 003 | Loss: 0.4326 | Val AUC: 0.8681 | Val AP: 0.4482


Evaluating: 100%|██████████| 2/2 [00:00<00:00,  3.23it/s]


Epoch 004 | Loss: 0.4077 | Val AUC: 0.8669 | Val AP: 0.4457


Evaluating: 100%|██████████| 2/2 [00:00<00:00,  3.32it/s]


Epoch 005 | Loss: 0.3863 | Val AUC: 0.8655 | Val AP: 0.4423

Best validation AUC: 0.8691


In [85]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    balanced_accuracy_score
)

import numpy as np


@torch.no_grad()
def evaluate_full(model, loader, device, threshold=0.5, top_k=100):
    """
    Full evaluation with multiple metrics
    """

    model.eval()

    all_preds = []
    all_labels = []

    for batch in tqdm(loader):
        batch = batch.to(device)

        z_chemical, z_disease = model(batch.x_dict, batch.edge_index_dict)

        edge_label_index = batch['chemical', 'chem_disease', 'disease'].edge_label_index
        edge_label = batch['chemical', 'chem_disease', 'disease'].edge_label

        pred = model.decode(z_chemical, z_disease, edge_label_index)
        pred = torch.sigmoid(pred)

        all_preds.append(pred.cpu())
        all_labels.append(edge_label.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    # =====================================================
    # Threshold-based metrics
    # =====================================================
    binary_preds = (all_preds >= threshold).astype(int)

    acc = accuracy_score(all_labels, binary_preds)
    precision = precision_score(all_labels, binary_preds)
    recall = recall_score(all_labels, binary_preds)
    f1 = f1_score(all_labels, binary_preds)
    bal_acc = balanced_accuracy_score(all_labels, binary_preds)

    # Specificity
    tn, fp, fn, tp = confusion_matrix(all_labels, binary_preds).ravel()
    specificity = tn / (tn + fp)

    # =====================================================
    # Ranking metrics
    # =====================================================
    auc = roc_auc_score(all_labels, all_preds)
    ap = average_precision_score(all_labels, all_preds)

    # =====================================================
    # Precision@K / Recall@K
    # =====================================================
    sorted_indices = np.argsort(-all_preds)
    top_k_indices = sorted_indices[:top_k]

    top_k_labels = all_labels[top_k_indices]

    precision_at_k = np.sum(top_k_labels) / top_k
    recall_at_k = np.sum(top_k_labels) / np.sum(all_labels)

    # =====================================================
    # Curves
    # =====================================================
    fpr, tpr, _ = roc_curve(all_labels, all_preds)
    precision_curve, recall_curve, _ = precision_recall_curve(all_labels, all_preds)

    results = {
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "Precision": precision,
        "Recall (Sensitivity)": recall,
        "Specificity": specificity,
        "F1 Score": f1,
        "ROC-AUC": auc,
        "PR-AUC": ap,
        f"Precision@{top_k}": precision_at_k,
        f"Recall@{top_k}": recall_at_k,
        "Confusion Matrix": {
            "TP": int(tp),
            "TN": int(tn),
            "FP": int(fp),
            "FN": int(fn)
        }
    }

    return results, fpr, tpr, precision_curve, recall_curve

# ============================================================================
# 11. FINAL EVALUATION
# ============================================================================
print("\n" + "="*80)
print("FINAL TEST EVALUATION (Full Metrics)")
print("="*80)

# model.load_state_dict(torch.load('best_model_minibatch.pt'))

results, fpr, tpr, pr_curve, rc_curve = evaluate_full(
    model,
    test_loader,
    device,
    threshold=0.5,
    top_k=200
)

for k, v in results.items():
    if k != "Confusion Matrix":
        print(f"{k}: {v:.4f}")
    else:
        print("\nConfusion Matrix:")
        for key, val in v.items():
            print(f"  {key}: {val}")



FINAL TEST EVALUATION (Full Metrics)


100%|██████████| 3/3 [00:00<00:00,  3.70it/s]


Accuracy: 0.9096
Balanced Accuracy: 0.8030
Precision: 0.5021
Recall (Sensitivity): 0.6726
Specificity: 0.9333
F1 Score: 0.5750
ROC-AUC: 0.9126
PR-AUC: 0.6130
Precision@200: 0.9800
Recall@200: 0.0004

Confusion Matrix:
  TP: 293788
  TN: 4076666
  FP: 291304
  FN: 143009


In [67]:


# ============================================================================
# 12. PREDICTION FUNCTION (Memory-Efficient)
# ============================================================================
def score_pair(z_chem, z_dis):
    edge_feat = torch.cat([z_chem, z_dis], dim=-1)
    return model.decoder(edge_feat)
@torch.no_grad()
def predict_top_k_batch(model, data, chemicals_list, diseases_list, k=10, batch_size=100, device='cpu'):
    model.eval()
    
    full_data = data.to(device)
    z_chemical, z_disease = model(full_data.x_dict, full_data.edge_index_dict)
    
    predictions = {}
    chem_items = list(chemicals_list.items())
    
    for i in tqdm(range(0, len(chem_items), batch_size), desc="Predicting"):
        batch_chems = chem_items[i:i+batch_size]
        
        for chem_name, chem_id in batch_chems:
            # Get chemical embedding (1, emb_dim)
            chem_emb = z_chemical[chem_id].unsqueeze(0)
            
            # Expand to match number of diseases -> (num_diseases, emb_dim)
            chem_emb_expanded = chem_emb.expand(z_disease.size(0), -1)
            
            # Concatenate along feature dimension -> (num_diseases, 2*emb_dim)
            edge_feat = torch.cat([chem_emb_expanded, z_disease], dim=-1)
            
            # Decoder outputs raw scores
            scores = model.decoder(edge_feat).squeeze()  # (num_diseases,)
            scores = torch.sigmoid(scores)
            
            # Get top-k diseases
            top_k_scores, top_k_indices = torch.topk(scores, k)
            
            # Map indices back to disease names
            disease_idx_to_name = {v: k for k, v in diseases_list.items()}
            top_diseases = [(disease_idx_to_name[idx.item()], score.item())
                            for idx, score in zip(top_k_indices, top_k_scores)]
            
            predictions[chem_name] = top_diseases
    
    return predictions


# Example predictions
print("\n" + "="*80)
print("STEP 11: Generating Predictions (Example)")
print("="*80)

first_5_chems = {k: v for k, v in list(chem_idx.items())[:5]}
predictions = predict_top_k_batch(model, data, first_5_chems, disease_idx, k=10, device=device)

print("\nTop-10 Drug-Disease Predictions for First 5 Chemicals:")
print("=" * 80)
for chem, disease_scores in predictions.items():
    print(f"\n{chem}:")
    for i, (disease, score) in enumerate(disease_scores, 1):
        print(f"  {i}. {disease}: {score:.4f}")

print("\n" + "="*80)
print("PIPELINE COMPLETE!")
print("="*80)
print("\nMemory optimizations applied:")
print("- Mini-batch training with LinkNeighborLoader")
print("- Neighbor sampling (10 neighbors in 1st hop, 5 in 2nd hop)")
print("- Batch size: 512 edges per batch")
print("- Reduced model dimensions (64 hidden, 32 output)")
print("- Gradient accumulation avoided")
print("\nIf you still encounter memory issues, you can further reduce:")
print("1. Batch size (currently 512 -> try 256 or 128)")
print("2. Number of neighbors (currently [10, 5] -> try [5, 3])")
print("3. Hidden dimensions (currently 64 -> try 32)")



STEP 11: Generating Predictions (Example)


Predicting: 100%|██████████| 1/1 [00:00<00:00,  6.24it/s]


Top-10 Drug-Disease Predictions for First 5 Chemicals:

C000015:
  1. C567384: 0.9740
  2. D010149: 0.9720
  3. D007024: 0.9687
  4. D004927: 0.9681
  5. 138900: 0.9667
  6. D014178: 0.9656
  7. D058387: 0.9570
  8. D016770: 0.9479
  9. D014123: 0.9423
  10. D059226: 0.9419

C000025:
  1. D011471: 0.0000
  2. D001943: 0.0000
  3. D006528: 0.0000
  4. D012559: 0.0000
  5. D006973: 0.0000
  6. D008175: 0.0000
  7. D003921: 0.0000
  8. D013274: 0.0000
  9. D008106: 0.0000
  10. D008114: 0.0000

C000050:
  1. D009181: 0.8928
  2. C565293: 0.8534
  3. D011251: 0.8524
  4. D016112: 0.8519
  5. D010383: 0.8483
  6. D012793: 0.8468
  7. D006486: 0.8464
  8. 615220: 0.8437
  9. D030342: 0.8391
  10. C562794: 0.8380

C000061:
  1. 614507: 0.9247
  2. 617412: 0.9227
  3. 612287: 0.9199
  4. D006053: 0.9133
  5. C567254: 0.9117
  6. C537451: 0.9079
  7. 617408: 0.9075
  8. 614900: 0.9056
  9. C567280: 0.9022
  10. C537934: 0.8957

C000081:
  1. D007806: 0.9776
  2. D007174: 0.9754
  3. C536545: 0